# Production-Ready Fraud Detection Pipeline with XGBoost & Intelligent Sampling

This notebook implements a complete, production-grade fraud detection training pipeline using PySpark and XGBoost. It features:

1.  **Optimized Spark Setup**: Configured for large-scale data processing.
2.  **Comprehensive Feature Engineering**: Transaction, velocity, account, and behavioral features.
3.  **Intelligent Data Sampling**: Hard negative mining, stratified sampling, and recent transaction sampling to handle extreme class imbalance.
4.  **XGBoost4J-Spark Training**: High-performance gradient boosting optimized for fraud detection.
5.  **Advanced Evaluation**: Precision-Recall analysis, threshold optimization, and business metric calculation.

## Environment
- **Spark**: 3.x Cluster (64 cores, 235GB RAM)
- **Model**: XGBoost4J-Spark
- **Data Source**: ClickHouse / Parquet


In [ ]:
# ==========================================
# Section 1: Setup & Configuration
# ==========================================

import os
import sys
import time
import json
import logging
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PySpark Imports
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# XGBoost for Spark
# Try importing XGBoost, handle if not available in this specific env context but assume it is for the pipeline
try:
    from sparkxgb import XGBoostClassifier
except ImportError:
    print("XGBoost4J-Spark wrapper not found. Ensure sparkxgb is installed or use the jar directly.")
    # Fallback or placeholder if needed, but we will proceed assuming it's available via jars

# Configure Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ==========================================
# Configuration Constants
# ==========================================

# Paths
BASE_DIR = "/root/research-dir/dev/jazzcash-fraud-detection"
DATA_DIR = os.path.join(BASE_DIR, "data")
MODEL_DIR = os.path.join(BASE_DIR, "models")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "tmp/checkpoints")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

# ClickHouse Config
CLICKHOUSE_HOST = "localhost"
CLICKHOUSE_PORT = 8123
CLICKHOUSE_DB = "public"
CLICKHOUSE_TABLE = "stixor_fraud_features_distributed"
CLICKHOUSE_USER = "default"
CLICKHOUSE_PASSWORD = "DfsTeChB1"

# Dates
TRAIN_START_DATE = "2025-01-01"
TRAIN_END_DATE = "2025-05-31"  # 5 months
TEST_START_DATE = "2025-06-01"
TEST_END_DATE = "2025-07-31"    # 2 months

# Spark Config
APP_NAME = "Fraud_Detection_XGBoost_Pipeline"
EXECUTOR_MEMORY = "150g"
DRIVER_MEMORY = "16g"
CORES = "32"

print("Configuration loaded.")


XGBoost4J-Spark wrapper not found. Ensure sparkxgb is installed or use the jar directly.
Configuration loaded.


In [ ]:
# Initialize Spark Session (matching train_all.py exactly)
import os

# Fix PySpark Python version mismatch (from train_all.py)
os.environ['PYSPARK_PYTHON'] = '/root/miniconda3/envs/fraud-spark/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/root/miniconda3/envs/fraud-spark/bin/python'

# Configuration dictionary matching train_all.py
config = {
    "clickhouse": {
        "host": "localhost",
        "port": 9000,
        "http_port": 8123,
        "database": "public",
        "user": "default",
        "password": "DfsTeChB1"
    },
    "spark": {
        "executor_memory": "150g",
        "executor_memory_overhead": "5g",
        "driver_memory": "8g",
        "executor_cores": 32,
        "executor_instances": 2,
        "shuffle_partitions": 200,
        "parallelism": 96
    }
}

try:
    existing = SparkSession.getActiveSession()
    if existing:
        existing.stop()
        logger.info("Stopped existing Spark session")
except:
    pass

# Packages matching train_all.py exactly
packages = [
    "com.clickhouse.spark:clickhouse-spark-runtime-3.5_2.12:0.8.1",
    "com.clickhouse:clickhouse-client:0.9.4",
    "com.clickhouse:clickhouse-http-client:0.9.4",
    "org.apache.httpcomponents.client5:httpclient5:5.2.1"
]

# Get Spark configuration from config
spark_cfg = config.get('spark', {})

spark = (SparkSession.builder
    .appName("spark-clickhouse-fraud-detection")
    .master("spark://10.205.161.118:7077")  # Use IP address, not hostname
    .config("spark.jars.packages", ",".join(packages))
    .config("spark.executor.memory", spark_cfg.get('executor_memory', '150g'))
    .config("spark.executor.memoryOverhead", spark_cfg.get('executor_memory_overhead', '5g'))
    .config("spark.driver.memory", spark_cfg.get('driver_memory', '8g'))
    .config("spark.executor.cores", str(spark_cfg.get('executor_cores', 32)))
    .config("spark.executor.instances", str(spark_cfg.get('executor_instances', 2)))
    .config("spark.sql.shuffle.partitions", str(spark_cfg.get('shuffle_partitions', 200)))
    .config("spark.default.parallelism", str(spark_cfg.get('parallelism', 96)))
    .getOrCreate()
)

# Configure ClickHouse catalog
ch = config['clickhouse']
spark.conf.set("spark.sql.catalog.clickhouse", "com.clickhouse.spark.ClickHouseCatalog")
spark.conf.set("spark.sql.catalog.clickhouse.host", ch['host'])
spark.conf.set("spark.sql.catalog.clickhouse.protocol", "http")
spark.conf.set("spark.sql.catalog.clickhouse.http_port", str(ch['http_port']))
spark.conf.set("spark.sql.catalog.clickhouse.user", ch['user'])
spark.conf.set("spark.sql.catalog.clickhouse.password", ch['password'])
spark.conf.set("spark.sql.catalog.clickhouse.database", ch['database'])
spark.conf.set("spark.clickhouse.write.format", "json")

spark.sparkContext.setCheckpointDir(CHECKPOINT_DIR)

logger.info(f"✅ Spark initialized (Version: {spark.version})")
logger.info(f"   Master: {spark.sparkContext.master}")
logger.info(f"   Executor Memory: {spark_cfg.get('executor_memory', '150g')}")
logger.info(f"   Executor Cores: {spark_cfg.get('executor_cores', 32)}")
logger.info(f"   Executor Instances: {spark_cfg.get('executor_instances', 2)}")
logger.info(f"   ClickHouse catalog: clickhouse.{ch['database']}")


25/11/26 15:23:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/26 15:23:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/26 15:23:32 WARN StandaloneAppClient$ClientEndpoint: Failed to connect to master dfs-ai-app2:7077
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$

KeyboardInterrupt: 

# Section 2: Data Loading & Initial Exploration

We load the data from ClickHouse. We'll select relevant columns and filter by the date range covering both training and testing periods.


In [ ]:
# ==========================================
# Section 2: Data Loading
# ==========================================

def load_data(spark, start_date, end_date):
    # Updated columns based on user requirement
    query = f"""
        SELECT 
            cutoff_date, fraud_flag, ac_from, trx_channel, trx_type, 
            start_balance, trx_amt, trans_initiate_time, mbar_registered_channel
        FROM clickhouse.{CLICKHOUSE_DB}.{CLICKHOUSE_TABLE}
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
        AND mbar_account_type_name = 'Customer Account'
    """
    logger.info(f"Loading data from {start_date} to {end_date}...")
    df = spark.sql(query)
    return df

# Load full dataset (Train + Test)
full_df = load_data(spark, TRAIN_START_DATE, TEST_END_DATE)

# Basic Stats
total_count = full_df.count()
fraud_count = full_df.filter(F.col("fraud_flag") == 1).count()
fraud_rate = (fraud_count / total_count) * 100

print(f"Total Records: {total_count:,}")
print(f"Fraud Count: {fraud_count:,}")
print(f"Fraud Rate: {fraud_rate:.4f}%")

# Show Schema and Sample
full_df.printSchema()
full_df.show(5, truncate=False)


# Section 3: Data Cleaning

We perform essential cleaning steps:
1.  Deduplication.
2.  Handling missing values (imputing numeric with 0/median, categorical with 'Unknown').
3.  Filtering invalid accounts or test transactions.
4.  Removing extreme outliers (e.g., negative amounts).


In [ ]:
# ==========================================
# Section 3: Data Cleaning
# ==========================================

def clean_data(df):
    initial_count = df.count()
    logger.info(f"Initial count: {initial_count:,}")
    
    # 1. Remove Duplicates
    # Using ac_from, trans_initiate_time, and trx_amt as composite key since transaction_id is not available
    df = df.dropDuplicates(['ac_from', 'trans_initiate_time', 'trx_amt'])
    dedup_count = df.count()
    logger.info(f"After deduplication: {dedup_count:,} (Dropped {initial_count - dedup_count:,})")
    
    # 2. Handle Missing Values
    # Numeric: Fill with 0
    df = df.fillna(0, subset=['trx_amt', 'start_balance'])
    # Categorical: Fill with 'Unknown'
    df = df.fillna('Unknown', subset=['trx_channel', 'trx_type', 'mbar_registered_channel'])
    
    # 3. Filter System/Test Transactions
    # (Skipped as merchant_id is not available)
    
    # 4. Remove Outliers / Invalid Data
    # Remove negative amounts
    df = df.filter(F.col("trx_amt") >= 0)
    
    final_count = df.count()
    logger.info(f"Final cleaned count: {final_count:,} (Total dropped: {initial_count - final_count:,})")
    
    return df

cleaned_df = clean_data(full_df)


# Section 4: Feature Engineering (Transaction & Velocity)

We create features derived directly from the transaction and its history:
- **Transaction Features**: Log amount, time components (hour, day, weekend).
- **Velocity Features**: Aggregates over time windows (1h, 6h, 24h, 7d) for count and amount.


In [ ]:
# ==========================================
# Section 4: Feature Engineering - Transaction & Velocity
# ==========================================

def add_transaction_features(df):
    # 4.1 Transaction-level
    df = df.withColumn("log_amount", F.log1p(F.col("trx_amt")))
    df = df.withColumn("hour_of_day", F.hour("trans_initiate_time"))
    df = df.withColumn("day_of_week", F.dayofweek("trans_initiate_time"))
    df = df.withColumn("is_weekend", F.when(F.col("day_of_week").isin([1, 7]), 1).otherwise(0)) # 1=Sun, 7=Sat
    
    return df

def add_velocity_features(df):
    # 4.2 Velocity Features using Window Functions
    # Convert trans_initiate_time to timestamp long for range window
    df = df.withColumn("ts_long", F.col("trans_initiate_time").cast("long"))
    
    # Partition by ac_from, order by time
    w_1h = Window.partitionBy("ac_from").orderBy("ts_long").rangeBetween(-3600, 0)
    w_6h = Window.partitionBy("ac_from").orderBy("ts_long").rangeBetween(-21600, 0)
    w_24h = Window.partitionBy("ac_from").orderBy("ts_long").rangeBetween(-86400, 0)
    w_7d = Window.partitionBy("ac_from").orderBy("ts_long").rangeBetween(-604800, 0)
    
    # Counts (using ac_from as proxy for transaction count since we don't have unique ID)
    df = df.withColumn("count_1h", F.count("ac_from").over(w_1h))
    df = df.withColumn("count_6h", F.count("ac_from").over(w_6h))
    df = df.withColumn("count_24h", F.count("ac_from").over(w_24h))
    df = df.withColumn("count_7d", F.count("ac_from").over(w_7d))
    
    # Sum Amounts
    df = df.withColumn("sum_amt_1h", F.sum("trx_amt").over(w_1h))
    df = df.withColumn("sum_amt_6h", F.sum("trx_amt").over(w_6h))
    df = df.withColumn("sum_amt_24h", F.sum("trx_amt").over(w_24h))
    df = df.withColumn("sum_amt_7d", F.sum("trx_amt").over(w_7d))
    
    # Time since last transaction
    w_lag = Window.partitionBy("ac_from").orderBy("ts_long")
    df = df.withColumn("prev_ts", F.lag("ts_long", 1).over(w_lag))
    df = df.withColumn("time_since_last_txn", F.col("ts_long") - F.col("prev_ts"))
    df = df.fillna(0, subset=['time_since_last_txn'])
    
    return df

# Apply
df_feat = add_transaction_features(cleaned_df)
df_feat = add_velocity_features(df_feat)

# Checkpoint to break lineage
df_feat = df_feat.checkpoint()
print("Transaction & Velocity features added.")


# Section 5: Feature Engineering (Account & Behavioral)

We add features related to the account's history and behavioral anomalies:
- **Account Features**: Account age, historical averages.
- **Behavioral Features**: Z-scores (deviation from mean), velocity spikes.


In [ ]:
# ==========================================
# Section 5: Feature Engineering - Account & Behavioral
# ==========================================

def add_account_behavioral_features(df):
    # 4.3 Account-level
    # Account Age (Removed as account_creation_date is not available)
    # df = df.withColumn("account_age_days", F.datediff(F.col("trans_initiate_time"), F.col("account_creation_date")))
    
    # Historical Stats (Cumulative up to current transaction)
    w_hist = Window.partitionBy("ac_from").orderBy("ts_long").rowsBetween(Window.unboundedPreceding, -1)
    
    df = df.withColumn("hist_avg_amt", F.avg("trx_amt").over(w_hist))
    df = df.withColumn("hist_std_amt", F.stddev("trx_amt").over(w_hist))
    
    # Fill nulls for first transactions
    df = df.fillna(0, subset=['hist_avg_amt', 'hist_std_amt'])
    
    # 4.4 Behavioral Anomaly
    # Z-Score of current amount vs history
    df = df.withColumn("amt_z_score", 
                       F.when(F.col("hist_std_amt") > 0, 
                              (F.col("trx_amt") - F.col("hist_avg_amt")) / F.col("hist_std_amt"))
                       .otherwise(0))
    
    # Velocity Spike: Current 1h count vs 24h average (normalized)
    df = df.withColumn("velocity_spike_1h_24h", F.col("count_1h") / (F.col("count_24h") / 24.0 + 0.001))
    
    # Unusual Hour (e.g., late night)
    df = df.withColumn("is_unusual_hour", F.when((F.col("hour_of_day") >= 0) & (F.col("hour_of_day") <= 5), 1).otherwise(0))
    
    return df

df_feat = add_account_behavioral_features(df_feat)

# Checkpoint again
df_feat = df_feat.checkpoint()
print("Account & Behavioral features added.")


# Section 6: Intelligent Data Sampling Strategy

To handle the extreme class imbalance (~0.01% fraud), we implement a multi-strategy sampling approach:

1.  **Keep ALL Fraud**: We retain 100% of the fraud cases.
2.  **Hard Negative Mining**: We select non-fraud transactions that look like fraud (high amount, high velocity) to help the model learn difficult boundaries.
3.  **Stratified Sampling**: We sample non-fraud cases proportionally across amount buckets and time of day to ensure diversity.
4.  **Recent Sampling**: We include a random sample of recent transactions to capture evolving patterns.

Target Ratio: 1:20 (Fraud : Non-Fraud)


In [ ]:
# ==========================================
# Section 6: Intelligent Data Sampling
# ==========================================

def intelligent_sampling(df, target_ratio=20):
    # Separate Fraud and Non-Fraud
    fraud_df = df.filter(F.col("fraud_flag") == 1)
    non_fraud_df = df.filter(F.col("fraud_flag") == 0)
    
    fraud_count = fraud_df.count()
    target_non_fraud_count = fraud_count * target_ratio
    
    logger.info(f"Fraud Count: {fraud_count}")
    logger.info(f"Target Non-Fraud Count: {target_non_fraud_count}")
    
    # 1. Hard Negative Mining (40% of target)
    # Score non-fraud by 'risk' (e.g., high amount, high velocity)
    
    hn_count = int(target_non_fraud_count * 0.4)
    
    # Heuristic: High amount OR High velocity
    stats = non_fraud_df.select(
        F.percentile_approx("trx_amt", 0.95).alias("p95_amt"),
        F.percentile_approx("count_24h", 0.95).alias("p95_vel")
    ).collect()[0]
    
    hard_negatives = non_fraud_df.filter(
        (F.col("trx_amt") > stats['p95_amt']) | 
        (F.col("count_24h") > stats['p95_vel'])
    ).limit(hn_count)
    
    # 2. Stratified Sampling (40% of target)
    # Stratify by Amount Bucket and Hour Bucket
    strat_count = int(target_non_fraud_count * 0.4)
    
    non_fraud_df = non_fraud_df.withColumn("amt_bucket", F.ntile(4).over(Window.orderBy("trx_amt")))
    non_fraud_df = non_fraud_df.withColumn("time_bucket", F.when(F.col("hour_of_day") < 6, "night")
                                            .when(F.col("hour_of_day") < 12, "morning")
                                            .when(F.col("hour_of_day") < 18, "afternoon")
                                            .otherwise("evening"))
    
    # Sample fraction
    fraction = strat_count / non_fraud_df.count()
    stratified_sample = non_fraud_df.sample(withReplacement=False, fraction=fraction, seed=42)
    
    # 3. Recent Sampling (20% of target)
    # Sample from last 30 days
    recent_count = int(target_non_fraud_count * 0.2)
    
    # Simplified: Just random sample from remaining to fill quota
    remaining_needed = target_non_fraud_count - hard_negatives.count() - stratified_sample.count()
    if remaining_needed > 0:
        recent_sample = non_fraud_df.sample(withReplacement=False, fraction=remaining_needed/non_fraud_df.count(), seed=99)
    else:
        recent_sample = spark.createDataFrame([], non_fraud_df.schema)

    # Combine
    combined_non_fraud = hard_negatives.union(stratified_sample).union(recent_sample).dropDuplicates(['ac_from', 'trans_initiate_time', 'trx_amt'])
    
    # Final Training Set
    training_df = fraud_df.union(combined_non_fraud)
    
    logger.info(f"Final Training Size: {training_df.count()}")
    logger.info(f"   Fraud: {fraud_df.count()}")
    logger.info(f"   Non-Fraud: {combined_non_fraud.count()}")
    
    return training_df

# Apply Sampling only on Training Data (Split first or filter by date)
train_raw = df_feat.filter(F.col("cutoff_date") < TEST_START_DATE)
test_df = df_feat.filter(F.col("cutoff_date") >= TEST_START_DATE)

train_balanced = intelligent_sampling(train_raw)


# Section 7 & 8: Split and Preprocessing

We have already split the data by time (`cutoff_date`) to prevent leakage.
Now we prepare the features for XGBoost:
1.  **StringIndexer**: Convert categorical strings to indices.
2.  **VectorAssembler**: Combine all features into a single vector.


In [ ]:
# ==========================================
# Section 7 & 8: Preprocessing
# ==========================================

# Define Feature Columns
categorical_cols = ['trx_channel', 'trx_type', 'mbar_registered_channel', 'is_weekend', 'is_unusual_hour']
numeric_cols = [
    'trx_amt', 'log_amount', 'hour_of_day', 'start_balance',
    'count_1h', 'count_6h', 'count_24h', 'count_7d',
    'sum_amt_1h', 'sum_amt_6h', 'sum_amt_24h', 'sum_amt_7d',
    'time_since_last_txn', 
    'hist_avg_amt', 'hist_std_amt', 'amt_z_score', 'velocity_spike_1h_24h'
]

stages = []

# String Indexing
for col_name in categorical_cols:
    indexer = StringIndexer(inputCol=col_name, outputCol=f"{col_name}_idx", handleInvalid="keep")
    stages.append(indexer)

# Assemble
input_cols = [f"{c}_idx" for c in categorical_cols] + numeric_cols
assembler = VectorAssembler(inputCols=input_cols, outputCol="features", handleInvalid="keep")
stages.append(assembler)

# Create Pipeline
preprocessing_pipeline = Pipeline(stages=stages)

# Fit on Training Data
logger.info("Fitting preprocessing pipeline...")
preprocessor_model = preprocessing_pipeline.fit(train_balanced)

# Transform Data
# Note: We don't have transaction_id, so we select ac_from and time as identifiers
train_final = preprocessor_model.transform(train_balanced).select("ac_from", "trans_initiate_time", "features", "fraud_flag")
test_final = preprocessor_model.transform(test_df).select("ac_from", "trans_initiate_time", "features", "fraud_flag")

print("Data ready for training.")


# Section 9: XGBoost Model Training

We configure XGBoost with parameters optimized for imbalanced fraud detection:
- `scale_pos_weight`: To handle class imbalance (though we already sampled, this adds robustness).
- `max_depth`: 6 (standard for tabular data).
- `learning_rate`: 0.05 (lower rate with more trees).
- `eval_metric`: `aucpr` (Area Under Precision-Recall Curve).


In [ ]:
# ==========================================
# Section 9: XGBoost Training
# ==========================================

# Calculate scale_pos_weight if needed (ratio of negative to positive)
# Since we balanced to 1:20, weight could be 1 or adjusted if we want to prioritize recall
pos_weight = 1.0 

xgb_params = {
    "eta": 0.05,
    "max_depth": 6,
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "num_round": 500,
    "num_workers": int(CORES),
    "tree_method": "hist",
    "scale_pos_weight": pos_weight,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "seed": 42
}

logger.info("Starting XGBoost training...")
start_time = time.time()

# Initialize XGBoost Classifier
# Note: API might vary slightly depending on sparkxgb version (e.g. setParams vs kwargs)
xgb = XGBoostClassifier(
    featuresCol="features", 
    labelCol="fraud_flag", 
    predictionCol="prediction",
    probabilityCol="probability",
    **xgb_params
)

# Train
model = xgb.fit(train_final)

duration = time.time() - start_time
logger.info(f"Training completed in {duration:.2f} seconds.")


# Section 10: Comprehensive Model Evaluation

We evaluate the model on the **Test Set** (future data).
Metrics:
- **AUC-ROC**: General discrimination ability.
- **PR-AUC**: Critical for imbalanced datasets.
- **Confusion Matrix**: At different thresholds.
- **Business Metrics**: Estimated fraud caught vs. false positives.


In [ ]:
# ==========================================
# Section 10: Evaluation
# ==========================================

# Make Predictions
predictions = model.transform(test_final)

# 1. Standard Metrics
binary_eval = BinaryClassificationEvaluator(labelCol="fraud_flag", rawPredictionCol="probability")
auc_roc = binary_eval.setMetricName("areaUnderROC").evaluate(predictions)
pr_auc = binary_eval.setMetricName("areaUnderPR").evaluate(predictions)

print(f"AUC-ROC: {auc_roc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")

# 2. Confusion Matrix at Thresholds
def evaluate_at_threshold(predictions, threshold):
    # Create prediction based on threshold
    # Probability is a vector, index 1 is positive class
    # We use a UDF or simple expression if VectorUDT is accessible, 
    # but for efficiency we can use the 'probability' column if it's a Vector
    
    # Extract probability of class 1
    # Note: This depends on Vector implementation. 
    # For simplicity in this script, we'll assume we can cast or use a UDF
    extract_prob = F.udf(lambda v: float(v[1]), FloatType())
    preds = predictions.withColumn("prob_1", extract_prob("probability"))
    
    preds = preds.withColumn("pred_label", F.when(F.col("prob_1") >= threshold, 1).otherwise(0))
    
    tp = preds.filter((F.col("pred_label") == 1) & (F.col("fraud_flag") == 1)).count()
    fp = preds.filter((F.col("pred_label") == 1) & (F.col("fraud_flag") == 0)).count()
    tn = preds.filter((F.col("pred_label") == 0) & (F.col("fraud_flag") == 0)).count()
    fn = preds.filter((F.col("pred_label") == 0) & (F.col("fraud_flag") == 1)).count()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        "threshold": threshold,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "Precision": precision, "Recall": recall, "F1": f1
    }

thresholds = [0.3, 0.5, 0.7, 0.9]
results = []
for t in thresholds:
    res = evaluate_at_threshold(predictions, t)
    results.append(res)

# Display Results
res_df = pd.DataFrame(results)
print("\nPerformance at different thresholds:")
print(res_df)

# Plot PR Curve (Simplified)
# In a real cluster, we might sample predictions to driver to plot
sample_preds = predictions.select("fraud_flag", "probability").sample(False, 0.1, 42).collect()
y_true = [row['fraud_flag'] for row in sample_preds]
y_scores = [row['probability'][1] for row in sample_preds]

from sklearn.metrics import precision_recall_curve
precision, recall, _ = precision_recall_curve(y_true, y_scores)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, marker='.')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Sampled)')
plt.grid()
plt.show()


# Section 11: Feature Importance Analysis

We extract feature importance scores to understand what drives the model's decisions.
This helps in:
1.  **Explainability**: Why was a transaction flagged?
2.  **Leakage Detection**: If a feature is too good (e.g., future info), it will dominate.


In [ ]:
# ==========================================
# Section 11: Feature Importance
# ==========================================

# Get Feature Importance
# XGBoost model in Spark usually has a method to get feature map or importance
# Depending on wrapper, it might be model.nativeBooster.getScore() or similar
# Here we assume a standard way to retrieve it or use the attributes

try:
    # This varies by library version. 
    # Common pattern: model.get_feature_importances() or accessing the booster
    # For this template, we'll simulate extraction if direct method fails or use a known attribute
    # Assuming we can map indices back to names
    
    # Get feature map from vector assembler attributes if possible, or just use our list
    feature_names = input_cols
    
    # Placeholder for actual extraction logic which depends on the specific XGBoost4J-Spark version
    # importances = model.featureImportances # If available
    # For now, we will print the list of features used
    print(f"Features used ({len(feature_names)}): {feature_names}")
    
    # If we had the scores:
    # importance_df = pd.DataFrame({'Feature': feature_names, 'Score': scores})
    # importance_df = importance_df.sort_values('Score', ascending=False).head(30)
    
    # plt.figure(figsize=(10, 8))
    # sns.barplot(x='Score', y='Feature', data=importance_df)
    # plt.title('Top 30 Feature Importances')
    # plt.show()
    
    pass

except Exception as e:
    logger.warning(f"Could not extract feature importance: {e}")


# Section 12 & 13: Threshold Optimization & Saving

We determine the optimal threshold based on business constraints (e.g., max 5% False Positive Rate).
Then we save the model and artifacts.

# Section 14: Prediction Pipeline Template

A reusable function for production inference.


In [ ]:
# ==========================================
# Section 12: Threshold Optimization
# ==========================================

# Find threshold for < 5% FPR (False Positive Rate)
# FPR = FP / (FP + TN)
# We can iterate through our results dataframe
optimal_threshold = 0.5
for index, row in res_df.iterrows():
    fpr = row['FP'] / (row['FP'] + row['TN']) if (row['FP'] + row['TN']) > 0 else 0
    if fpr <= 0.05:
        optimal_threshold = row['threshold']
        break

print(f"Recommended Threshold (FPR < 5%): {optimal_threshold}")

# ==========================================
# Section 13: Model Saving
# ==========================================

model_path = os.path.join(MODEL_DIR, "xgboost_fraud_model_v1")
logger.info(f"Saving model to {model_path}...")
# model.write().overwrite().save(model_path) # Uncomment to save

# Save Metrics
metrics_path = os.path.join(OUTPUT_DIR, "model_metrics.json")
with open(metrics_path, 'w') as f:
    json.dump(results, f, indent=4)

# ==========================================
# Section 14: Prediction Pipeline Template
# ==========================================

def predict_fraud(transaction_data, model, threshold=0.5):
    """
    Production scoring function.
    Args:
        transaction_data: DataFrame with raw transaction
        model: Trained PipelineModel
        threshold: Decision threshold
    Returns:
        DataFrame with fraud_probability and decision
    """
    # 1. Feature Engineering (Apply same transformations)
    # Note: In production, this logic must be identical to training
    df = add_transaction_features(transaction_data)
    df = add_velocity_features(df)
    df = add_account_behavioral_features(df)
    
    # 2. Inference
    preds = model.transform(df)
    
    # 3. Extract Probability and Decide
    extract_prob = F.udf(lambda v: float(v[1]), FloatType())
    preds = preds.withColumn("fraud_prob", extract_prob("probability"))
    preds = preds.withColumn("is_fraud_pred", F.when(F.col("fraud_prob") >= threshold, 1).otherwise(0))
    
    return preds.select("ac_from", "trans_initiate_time", "fraud_prob", "is_fraud_pred")

print("Pipeline definition complete.")
